In [3]:
import pandas as pd
import numpy as np
import scipy.sparse as sparse
import implicit

In [ ]:
# interactions = pd.read_csv(r'interactions_train.csv')
from pathlib import Path
interactions = pd.read_csv(Path.cwd().parent/"data"/"interactions_train.csv")

In [4]:
interactions

,u,i,t
0,4456,8581,1.687541e+09
1,142,1964,1.679585e+09
2,362,3705,1.706872e+09
3,1809,11317,1.673533e+09
4,4384,1323,1.681402e+09
...,...,...,...
87042,924,8171,1.699284e+09
87043,1106,9009,1.699872e+09
87044,5207,13400,1.683627e+09
87045,698,5779,1.686667e+09


In [ ]:
# Matrix Factorization requires contiguous integer IDs starting from 0.
print("Mapping IDs")
interactions['user_id_cat'] = interactions['u'].astype('category')
interactions['item_id_cat'] = interactions['i'].astype('category')

user_cat_codes = interactions['user_id_cat'].cat.codes
item_cat_codes = interactions['item_id_cat'].cat.codes

# Save the mappings so we can reverse them later for the submission file
user_id_map = dict(enumerate(interactions['user_id_cat'].cat.categories))
item_id_map = dict(enumerate(interactions['item_id_cat'].cat.categories))

Mapping IDs...


In [6]:
# Implicit needs a sparse matrix to represent interactions.
# We assign a 'weight' or 'confidence' of 1 to every interaction.
print("Building sparse matrix")
confidences = np.ones(len(interactions))

# Create a sparse CSR matrix of shape (num_users, num_items)
user_items_sparse = sparse.csr_matrix(
    (confidences, (user_cat_codes, item_cat_codes)),
    shape=(len(user_id_map), len(item_id_map))
)

Building sparse matrix


In [7]:
# Initialize and Train the ALS Model
print("Training ALS model")
# Hyperparameters: 
# factors: the size of the latent dimensions. 64 or 128 is a good start.
# regularization: helps prevent overfitting.
# iterations: number of passes over the data.
model = implicit.als.AlternatingLeastSquares(
    factors=64, 
    regularization=0.1, 
    iterations=20,
    random_state=42
)

# Fit the model. Note: Recent versions of 'implicit' expect user x item matrices.
model.fit(user_items_sparse)

Training ALS model


e:\anaconda3\envs\datascience_machinelearning\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
e:\anaconda3\envs\datascience_machinelearning\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [18]:
# Generate Top-10 Recommendations
print("Generating top-10 recommendations for all users")

# Create an array of all user indices
all_user_indices = np.arange(user_items_sparse.shape[0])

# Pass the array of users to the standard recommend function
# This returns a tuple: (2D array of item indices, 2D array of scores)
recommendations, scores = model.recommend(
    userid=all_user_indices,
    user_items=user_items_sparse,
    N=10, 
    filter_already_liked_items=True
)

Generating top-10 recommendations for all users


In [19]:
print("Formatting submission file")
submission_data = []

# recommendations is now a 2D numpy array of shape (num_users, 10)
for user_idx, item_indices in enumerate(recommendations):
    original_user_id = user_id_map[user_idx]
    
    # Map the 10 internal item indices back to their original item IDs
    original_item_ids = [str(item_id_map[idx]) for idx in item_indices]
    
    # Join the top 10 items into a single space-separated string
    top_10_string = " ".join(original_item_ids)
    
    submission_data.append({
        'user_id': original_user_id,
        'recommendations': top_10_string
    })

submission_df = pd.DataFrame(submission_data)
submission_df.to_csv('als_baseline_submission_1.csv', index=False)
print("Baseline submission saved to 'als_baseline_submission_1.csv'.")

Formatting submission file
Baseline submission saved to 'als_baseline_submission_1.csv'.
